# ASSN2 — vLLM + OpenRLHF for Efficient Math Reasoning

This notebook trains and evaluates preference-optimized math models with **OpenRLHF**, **vLLM**, and **Hugging Face**.

**TA:** Minju Gwak (mjgwak@yonsei.ac.kr)

## Goal

Run the provided SFT + DPO pipeline, then improve it by editing only sections marked:

```text
### EDIT HERE ###
```

Do not modify setup, evaluation, merge, or upload cells unless explicitly instructed.

## What To Edit

- **SFT:** learning rate only.
- **Preference data:** prompt, model, decoding settings, filters, and pair construction. (You can have freedom over the entire cell)
- **DPO:** learning rate, beta, train batch size, and micro batch size only.
- **SimPO:** implement the algorithm and add its training/evaluation pipeline after the provided cells.

## Grading

- Environment and notebook execution: **10%**
- Hugging Face upload + leaderboard submission with SFT/DPO: **10%**
- Performance improvement from your configuration choices: **10%**
- Original synthetic preference-data pipeline: **15%**
- SimPO implementation on OpenRLHF + successful training/evaluation/upload: **25%**
- Analysis and justification of data generation and algorithm choices: **30%**

## Pipeline

1. **Setup:** install editable OpenRLHF and initialize paths. (Refer to README.md)
2. **Hugging Face login:** authenticate for gated models and uploads.
3. **SFT:** train a LoRA adapter, then merge it into an HF model.
4. **Preference data:** generate `chosen`/`rejected` pairs with vLLM.
5. **DPO:** train on the generated preference data.
6. **Evaluation:** use the shared evaluation harness.
7. **Upload:** upload one final dataset/model to Hugging Face and the leaderboard.
8. **SimPO:** implement in OpenRLHF, train, evaluate, and upload your SimPO model.

## Datasets and Model Used in the Current Notebook

| Stage | Source / Model | Local file or checkpoint | Purpose |
|---|---|---|---|
| Base model | `meta-llama/Llama-3.2-1B-Instruct` by default | `MODEL_BASE_ID` | Starting model for SFT, DPO, and evaluation. |
| SFT data | `openai/gsm8k` (`main`, train split) | `data/train.jsonl` | Supervised fine-tuning data with `problem` / `solution` fields. |
| Preference data | `talzoomanzoo/gsm8k` (preprocessed from `openai/gsam8k`) + vLLM generations | `data/assn2_pref_dpo.jsonl` | DPO/SimPO pairs with `prompt`, `chosen`, and `rejected`. |
| Label data | Same vLLM generations as preference data | `data/assn2_pref_labels.jsonl` | Optional positive/negative response labels for analysis. |
| Evaluation data | Course-provided eval JSONL | `data/assn2_eval.jsonl` | Held-out evaluation set for students |
| SFT checkpoint | OpenRLHF LoRA output | `checkpoints/assn2-sft` | Adapter checkpoint from SFT. |
| Merged SFT model | Base model + SFT LoRA | `checkpoints/assn2-sft-merged-hf` | Input model for DPO/SimPO. |
| DPO checkpoint | OpenRLHF DPO output | `checkpoints/assn2-dpo` | Preference-tuned model. |

## Notes

- Expected runtime on the current setup: SFT ~25 min on 1x RTX 3090; preference optimization ~10 min on 4x RTX 3090.
- Get Hugging Face access approval for `meta-llama/Llama-3.2-1B-Instruct` before running model cells.
- If CUDA memory remains allocated after a failed run, restart the kernel.
- Merge LoRA adapters before evaluation/upload.
- Submit only one final trained model to the leaderboard (https://huggingface.co/spaces/talzoomanzoo/alignment-leaderboard).
- Leaderboard performance may differ from your held-out evaluation set because it is built with a different datsaet.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
import os
import re
import subprocess
import sys
from pathlib import Path
from typing import Iterable
import openrlhf

PROJECT_ROOT = Path.cwd()
THIRD_PARTY   = PROJECT_ROOT / "third_party"
DATA_DIR      = PROJECT_ROOT / "data"
CKPT_DIR      = PROJECT_ROOT / "checkpoints"

MODEL_BASE_ID = os.environ.get("ASSN2_BASE_MODEL", "meta-llama/Llama-3.2-1B-Instruct")

JSONL_SFT_TRAIN   = DATA_DIR / "train.jsonl"
JSONL_PREF_DPO    = DATA_DIR / "assn2_pref_dpo.jsonl"
JSONL_PREF_LABELS = DATA_DIR / "assn2_pref_labels.jsonl"
JSONL_EVAL        = DATA_DIR / "assn2_eval.jsonl"

CKPT_SFT_LORA     = CKPT_DIR / "assn2-sft"
CKPT_SFT_MERGED   = CKPT_DIR / "assn2-sft-merged-hf"
CKPT_DPO          = CKPT_DIR / "assn2-dpo"
CKPT_DPO_MERGED   = CKPT_DIR / "assn2-dpo-merged-hf"
CKPT_SIMPO        = CKPT_DIR / "assn2-simpo"
CKPT_SIMPO_MERGED = CKPT_DIR / "assn2-simpo-merged-hf"

print("MODEL_BASE_ID:", MODEL_BASE_ID)

In [ ]:
THIRD_PARTY.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

os.environ.setdefault("HF_HOME",             str(PROJECT_ROOT / ".hf"))
os.environ.setdefault("TRANSFORMERS_CACHE",  str(PROJECT_ROOT / ".hf" / "transformers"))
os.environ.setdefault("HF_DATASETS_CACHE",  str(PROJECT_ROOT / ".hf" / "datasets"))

print("python:", sys.version)
print("project_root:", PROJECT_ROOT)

try:
    import torch
    print("torch:", torch.__version__)
    print("cuda_available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("gpu:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch not available yet:", e)

In [ ]:
# Writes build_prompt / extract_final into the editable OpenRLHF install.
# ### EDIT HERE ### modify build_prompt or extract_final to improve reasoning format.

import openrlhf

OPENRLHF_PKG_DIR = Path(openrlhf.__file__).resolve().parent
ASSN2_PKG_DIR    = OPENRLHF_PKG_DIR / "assn2"
ASSN2_PKG_DIR.mkdir(parents=True, exist_ok=True)

(ASSN2_PKG_DIR / "__init__.py").write_text(
    '"""ASSN2 helpers that live inside OpenRLHF (editable install)."""\n', encoding="utf-8"
)

(ASSN2_PKG_DIR / "reasoning.py").write_text(
    r'''from __future__ import annotations
import re

_FINAL_RE = re.compile(r"####\s*([-+]?\d+(?:\.\d+)?)")


def build_prompt(question: str) -> str:
    """A simple, consistent reasoning prompt used by this assignment."""
    return (
        "You are a careful reasoner. Solve step-by-step. "
        "At the end, output the final answer on its own line as: \'#### <answer>\'\n\n"
        f"Question: {question}\n\nAnswer:\n"
    )


def extract_final(text: str) -> str | None:
    """Extract the \'#### <answer>\' final answer token from a model output."""
    m = None
    for m in _FINAL_RE.finditer(text):
        pass
    return m.group(1) if m else None
''', encoding="utf-8"
)

from openrlhf.assn2.reasoning import build_prompt, extract_final

print("Wrote:", ASSN2_PKG_DIR / "reasoning.py")
print("Prompt example:\n", build_prompt("What is 2+2?"))
print("Extract example:", extract_final("blah\n#### 4\n"))

## 2. Hugging Face Login

In [ ]:
import os
from getpass import getpass

try:
    from huggingface_hub import login
except Exception:
    %pip install -U huggingface_hub
    from huggingface_hub import login

env_token = (os.environ.get("HF_TOKEN") or "").strip()
if env_token:
    use_env = input("HF_TOKEN is set. Use it? [Y/n]: ").strip().lower()
    token = env_token if use_env in ("", "y", "yes") else getpass("Enter HF token: ").strip()
else:
    token = getpass("Enter HF token: ").strip()

if not token:
    raise ValueError("No token provided.")

login(token=token)
os.environ["HF_TOKEN"] = token
print("Hugging Face login OK.")

## 3. Evaluation Harness

In [ ]:
import json
import re
from datasets import load_dataset

_FINAL_RE_EVAL = re.compile(r"####\s*([-+]?\d+(?:\.\d+)?)")

def norm_aime_answer(x) -> str:
    if x is None:
        return ""
    s = str(x).strip()
    m = _FINAL_RE_EVAL.search(s)
    if m:
        s = m.group(1)
    try:
        return str(int(round(float(s))))
    except (TypeError, ValueError, OverflowError):
        return s

DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Loading talzoomanzoo/gsm8k (test split) ...")

ds_eval = (
    load_dataset("talzoomanzoo/gsm8k", split="test")
    .shuffle(seed=42)
    .select(range(100))
)

with JSONL_EVAL.open("w", encoding="utf-8") as f:
    for row in ds_eval:
        gold = norm_aime_answer(row["answer"])
        f.write(json.dumps({"question": row["Question"], "answer": gold}, ensure_ascii=False) + "\n")

print("Wrote", JSONL_EVAL, "—", len(ds_eval), "examples")

In [ ]:
import math
import re
from pathlib import Path

_BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")

def build_eval_prompt(question: str) -> str:
    return (
        "Please answer the following math question.\n"
        "You should provide your final answer in the format\n"
        "\\boxed{YOUR_ANSWER}.\n\n"
        f"Question:\n{question}\n"
    )

def extract_final_eval(text: str) -> str | None:
    if not text:
        return None
    boxed = list(_BOXED_RE.finditer(text))
    if boxed:
        return boxed[-1].group(1).strip()
    from openrlhf.assn2.reasoning import extract_final as extract_final_hash
    return extract_final_hash(text)

def norm_eval_answer(x) -> str:
    if x is None:
        return ""
    try:
        fx = float(x)
        if not math.isfinite(fx):
            return str(x).strip()
        return str(int(round(fx)))
    except (TypeError, ValueError, OverflowError):
        return str(x).strip()

def load_eval_examples(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines()]

eval_examples = load_eval_examples(JSONL_EVAL)
print(f"Loaded {len(eval_examples)} eval examples")

In [ ]:
import gc
import json
import os
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

def _eval_model_hf(model_name_or_path: str, examples: list[dict], max_new_tokens: int, mp: Path) -> float:
    adapter_cfg_path = mp / "adapter_config.json"
    if adapter_cfg_path.exists():
        adapter_cfg = json.loads(adapter_cfg_path.read_text(encoding="utf-8"))
        base_id = adapter_cfg.get("base_model_name_or_path") or MODEL_BASE_ID
        if not torch.cuda.is_available():
            raise RuntimeError("QLoRA adapter evaluation needs CUDA.")
        eval_gpu = int(os.environ.get("ASSN2_EVAL_GPU", "0"))
        quant_cfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
        model = AutoModelForCausalLM.from_pretrained(base_id, trust_remote_code=True,
                                                      quantization_config=quant_cfg, device_map={"": eval_gpu})
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(mp))
        tok_src = mp if (mp / "tokenizer_config.json").exists() else base_id
        tok = AutoTokenizer.from_pretrained(tok_src, use_fast=True)
    else:
        if torch.cuda.is_available():
            eval_gpu = int(os.environ.get("ASSN2_EVAL_GPU", "0"))
            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            device_map = {"": eval_gpu}
        else:
            dtype = torch.float32
            device_map = {"": "cpu"}
        tok = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(model_name_or_path, trust_remote_code=True,
                                                       torch_dtype=dtype, device_map=device_map)

    model.eval()
    correct = 0
    for ex in examples:
        prompt = build_eval_prompt(ex["question"])
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        prompt_len = inputs["input_ids"].shape[1]
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0)
        text = tok.decode(out[0, prompt_len:], skip_special_tokens=True)
        pred = extract_final_eval(text)
        gold = norm_eval_answer(ex["answer"])
        pred_n = norm_eval_answer(pred) if pred is not None else ""
        ok = pred_n == gold
        correct += int(ok)
        print(f"Q: {ex['question']}\npred: {pred_n or pred}  gold: {gold}  ok: {ok}\n---")

    acc = correct / max(1, len(examples))
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    return acc

In [ ]:
import gc
import importlib.util
import os
import torch
from pathlib import Path

def _cleanup_dist():
    import torch.distributed as dist
    try:
        import vllm.distributed.parallel_state as _ps
    except ImportError:
        _ps = None
    if _ps is not None:
        try: _ps.destroy_model_parallel()
        except Exception: pass
        for attr in ("_WORLD", "_INNER_DP_WORLD"):
            obj = getattr(_ps, attr, None)
            if obj is not None:
                try: obj.destroy()
                except Exception: pass
                setattr(_ps, attr, None)
        try: _ps._NODE_COUNT = None
        except Exception: pass
    try:
        if dist.is_initialized():
            dist.destroy_process_group()
    except Exception: pass

def _eval_model_vllm(model_name_or_path: str, examples: list[dict], max_new_tokens: int, mp: Path) -> float:
    import json
    os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
    if os.environ.get("ASSN2_VLLM_DISABLE_FLASHINFER_SAMPLER", "").strip().lower() in ("1","true","yes"):
        os.environ.setdefault("VLLM_USE_FLASHINFER_SAMPLER", "0")
    if os.environ.get("ASSN2_VLLM_ENABLE_V1_MP", "0").strip().lower() not in ("1","true","yes"):
        os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

    if not torch.cuda.is_available():
        raise RuntimeError("vLLM evaluation requires CUDA. Set ASSN2_EVAL_BACKEND=hf for CPU.")

    _cleanup_dist()

    from vllm import LLM, SamplingParams
    from vllm.config.attention import AttentionConfig
    from vllm.lora.request import LoRARequest
    from vllm.v1.attention.backends.registry import AttentionBackendEnum

    vllm_kw = {
        "trust_remote_code": True, "dtype": "auto",
        "tensor_parallel_size": max(1, int(os.environ.get("ASSN2_VLLM_TP", "1"))),
        "gpu_memory_utilization": float(os.environ.get("ASSN2_VLLM_GPU_MEM_UTIL", "0.88")),
        "max_model_len": int(os.environ.get("ASSN2_VLLM_MAX_MODEL_LEN", "32768")),
        "disable_custom_all_reduce": True,
    }
    if os.environ.get("ASSN2_VLLM_ENFORCE_EAGER", "1").strip().lower() not in ("0","false","no"):
        vllm_kw["enforce_eager"] = True

    name = os.environ.get("ASSN2_VLLM_ATTENTION_BACKEND", "TRITON_ATTN").strip().upper()
    if name and name != "AUTO":
        try:
            vllm_kw["attention_config"] = AttentionConfig(backend=AttentionBackendEnum[name])
        except KeyError as err:
            opts = ", ".join(sorted(AttentionBackendEnum.__members__.keys()))
            raise ValueError(f"Unknown backend {name!r}. Options: {opts}") from err

    adapter_cfg_path = mp / "adapter_config.json"
    lora_req = None
    if adapter_cfg_path.exists():
        adapter_cfg = json.loads(adapter_cfg_path.read_text(encoding="utf-8"))
        base_id = adapter_cfg.get("base_model_name_or_path") or MODEL_BASE_ID
        rank = int(adapter_cfg.get("r", 64))
        vllm_kw["model"] = base_id
        vllm_kw["enable_lora"] = True
        vllm_kw["max_lora_rank"] = max(rank, 8)
        vllm_kw["max_loras"] = max(1, int(os.environ.get("ASSN2_EVAL_VLLM_MAX_LORAS", "4")))
        lora_req = LoRARequest("assn2_adapter", 1, str(mp.resolve()))
    else:
        vllm_kw["model"] = str(model_name_or_path)

    llm = LLM(**vllm_kw)
    sampling_params = SamplingParams(temperature=0.0, max_tokens=max_new_tokens)
    bs = max(1, int(os.environ.get("ASSN2_EVAL_VLLM_BATCH_SIZE", "16")))
    correct = 0

    for start in range(0, len(examples), bs):
        chunk = examples[start: start + bs]
        prompts = [build_eval_prompt(ex["question"]) for ex in chunk]
        outs = llm.generate(prompts, sampling_params, lora_request=lora_req) if lora_req else llm.generate(prompts, sampling_params)
        for ex, req_out in zip(chunk, outs):
            text = req_out.outputs[0].text
            pred = extract_final_eval(text)
            gold = norm_eval_answer(ex["answer"])
            pred_n = norm_eval_answer(pred) if pred is not None else ""
            ok = pred_n == gold
            correct += int(ok)
            print(f"Q: {ex['question']}\npred: {pred_n or pred}  gold: {gold}  ok: {ok}\n---")

    acc = correct / max(1, len(examples))
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    _cleanup_dist()
    return acc

In [ ]:
import os
from pathlib import Path

def eval_model(model_name_or_path: str, examples: list[dict], max_new_tokens: int | None = None) -> float:
    mp = Path(model_name_or_path)
    if max_new_tokens is None:
        max_new_tokens = int(os.environ.get("ASSN2_EVAL_MAX_NEW", "768"))
    backend = os.environ.get("ASSN2_EVAL_BACKEND", "vllm").strip().lower()
    if backend == "hf":
        return _eval_model_hf(model_name_or_path, examples, max_new_tokens, mp)
    try:
        import vllm  # noqa: F401
    except Exception as exc:
        raise RuntimeError(f"vllm not importable: {exc!r}") from exc
    return _eval_model_vllm(model_name_or_path, examples, max_new_tokens, mp)

print("eval_model dispatcher ready")

## 4. Base Model Evaluation

In [ ]:
base_acc = eval_model(MODEL_BASE_ID, eval_examples)
print("MODEL_BASE_ID:", MODEL_BASE_ID)
print("base_acc:", base_acc)

## 5. Preference Data Generation (vLLM)

> **EDIT HERE** — you may freely edit this entire section.

In [ ]:
import importlib.util
import os

def _set_env_default(name: str, value: str) -> None:
    if os.environ.get(name) is None:
        get_ipython().run_line_magic("env", f"{name}={value}")

_set_env_default("VLLM_WORKER_MULTIPROC_METHOD", "spawn")
_set_env_default("ASSN2_VLLM_DISABLE_FLASHINFER_SAMPLER", "1")
if os.environ.get("ASSN2_VLLM_DISABLE_FLASHINFER_SAMPLER", "").strip().lower() not in ("0","false","no"):
    _set_env_default("VLLM_USE_FLASHINFER_SAMPLER", "0")
if os.environ.get("ASSN2_VLLM_ENABLE_V1_MP", "0").strip().lower() not in ("1","true","yes"):
    get_ipython().run_line_magic("env", "VLLM_ENABLE_V1_MULTIPROCESSING=0")

# Remove broken flash-attn installs that crash vLLM imports.
if os.environ.get("ASSN2_UNINSTALL_FLASH_ATTN_FOR_VLLM", "1").strip().lower() not in ("0","false","no"):
    if importlib.util.find_spec("flash_attn") is not None:
        try:
            import flash_attn.flash_attn_interface  # noqa: F401
        except ImportError as _e:
            _em = str(_e).lower()
            if "flash_attn_2_cuda" in _em or "undefined symbol" in _em:
                get_ipython().run_line_magic("pip", "uninstall -y flash-attn")
                raise RuntimeError(
                    "Uninstalled incompatible flash-attn (PyTorch ABI mismatch). "
                    "Restart the kernel, then re-run from the setup cell."
                ) from _e
            raise

%pip install -q vllm
print("vLLM env ready")

In [ ]:
# ### EDIT HERE ### tweak sampling parameters and dataset size.
MAX_SEED   = 1000   # how many GSM8K train examples to sample from
NUM_SAMPLES = 8     # generations per question
GEN_TEMP    = 0.9
MAX_NEW     = 4096

VLLM_BATCH        = int(os.environ.get("ASSN2_PREF_VLLM_BATCH", "20"))
VLLM_GPU_MEM      = float(os.environ.get("ASSN2_VLLM_GPU_MEM_UTIL", "0.75"))
VLLM_TP           = int(os.environ.get("ASSN2_VLLM_TP", "1"))
VLLM_MAX_MODEL_LEN = int(os.environ.get("ASSN2_VLLM_MAX_MODEL_LEN", "4096"))
MAX_DPO_PAIRS     = int(os.environ.get("ASSN2_MAX_DPO_PAIRS", "1000"))

print(f"MAX_SEED={MAX_SEED}  NUM_SAMPLES={NUM_SAMPLES}  GEN_TEMP={GEN_TEMP}  MAX_DPO_PAIRS={MAX_DPO_PAIRS}")

In [ ]:
import gc
import random
import torch

def norm_answer(x):
    if x is None:
        return ""
    try:
        return str(int(round(float(x))))
    except Exception:
        return str(x).strip()

def fallback_positive(ans):
    a = norm_answer(ans)
    return f"Step-by-step reasoning.\nFinal answer: {a}\n#### {a}"

def fallback_negative(ans):
    a = norm_answer(ans)
    wrong = str(int(a) + 3) if a.lstrip("-").isdigit() else "0"
    return f"Quick guess.\nFinal answer: {wrong}\n#### {wrong}"

def load_vllm_engine():
    from vllm import LLM
    from vllm.config.attention import AttentionConfig
    from vllm.v1.attention.backends.registry import AttentionBackendEnum

    kw = {
        "model": MODEL_BASE_ID, "trust_remote_code": True, "dtype": "auto",
        "tensor_parallel_size": VLLM_TP, "gpu_memory_utilization": VLLM_GPU_MEM,
        "max_model_len": VLLM_MAX_MODEL_LEN, "disable_custom_all_reduce": True,
        "disable_log_stats": True,
    }
    if os.environ.get("ASSN2_VLLM_ENFORCE_EAGER", "1").strip().lower() not in ("0","false","no"):
        kw["enforce_eager"] = True

    name = os.environ.get("ASSN2_VLLM_ATTENTION_BACKEND", "TRITON_ATTN").strip().upper()
    if name and name != "AUTO":
        try:
            kw["attention_config"] = AttentionConfig(backend=AttentionBackendEnum[name])
        except KeyError as err:
            opts = ", ".join(sorted(AttentionBackendEnum.__members__.keys()))
            raise ValueError(f"Unknown backend {name!r}. Options: {opts}") from err

    _cleanup_dist()
    return LLM(**kw)

print("Helpers defined")

In [ ]:
from datasets import load_dataset

print("Loading GSM8K train split...")
ds = load_dataset("talzoomanzoo/gsm8k", split="train").shuffle(seed=42).select(range(min(MAX_SEED, 7473)))
rows = list(ds)
print(f"Loaded {len(rows)} examples")

In [ ]:
from vllm import SamplingParams
from openrlhf.assn2.reasoning import build_prompt, extract_final

print(f"Loading vLLM engine: {MODEL_BASE_ID}")
llm = load_vllm_engine()

sampling_params = SamplingParams(n=NUM_SAMPLES, temperature=GEN_TEMP, top_p=0.95, max_tokens=MAX_NEW)

label_rows = []
dpo_rows   = []
stop_all   = False

for start in range(0, len(rows), VLLM_BATCH):
    if stop_all:
        break
    batch   = rows[start: start + VLLM_BATCH]
    prompts = [build_prompt(ex["Question"]) for ex in batch]
    outputs = llm.generate(prompts, sampling_params)

    for ex, prompt_text, out in zip(batch, prompts, outputs):
        if stop_all:
            break
        gold  = norm_answer(ex["answer"])
        comps = [o.text for o in out.outputs]

        positives = [c for c in comps if norm_answer(extract_final(c)) == gold]
        negatives = [c for c in comps if norm_answer(extract_final(c)) not in ("", gold)]

        for p in (positives or [fallback_positive(gold)]):
            label_rows.append({"prompt": prompt_text, "response": p, "label": True,  "answer": gold})
        for n in (negatives or [fallback_negative(gold)]):
            label_rows.append({"prompt": prompt_text, "response": n, "label": False, "answer": gold})

        if positives and negatives:
            if len(dpo_rows) >= MAX_DPO_PAIRS:
                stop_all = True
                break
            dpo_rows.append({"prompt": prompt_text, "chosen": random.choice(positives),
                              "rejected": random.choice(negatives), "answer": gold})

del llm
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
_cleanup_dist()
print(f"Generated {len(dpo_rows)} DPO pairs, {len(label_rows)} label rows")

In [ ]:
import json

with JSONL_PREF_DPO.open("w", encoding="utf-8") as f:
    for r in dpo_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"DPO pairs  ({len(dpo_rows)}) -> {JSONL_PREF_DPO}")

with JSONL_PREF_LABELS.open("w", encoding="utf-8") as f:
    for r in label_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Label rows ({len(label_rows)}) -> {JSONL_PREF_LABELS}")

## 6. SFT Training

In [ ]:
import json
import os
from datasets import load_dataset

MAX_SAMPLES = int(os.environ.get("ASSN2_SFT_MAX_SAMPLES", "5000"))

ds_sft = load_dataset("openai/gsm8k", "main", split="train").shuffle(seed=42)
if MAX_SAMPLES > 0:
    ds_sft = ds_sft.select(range(min(MAX_SAMPLES, len(ds_sft))))

def to_boxed_solution(answer_text: str) -> str:
    reasoning, sep, final = answer_text.rpartition("####")
    if not sep:
        return answer_text
    return reasoning.rstrip() + f"\n\\boxed{{{final.strip()}}}"

CKPT_SFT_LORA.mkdir(parents=True, exist_ok=True)
with open(JSONL_SFT_TRAIN, "w", encoding="utf-8") as f:
    for ex in ds_sft:
        f.write(json.dumps({"problem": ex["question"], "solution": to_boxed_solution(ex["answer"])},
                            ensure_ascii=False) + "\n")

print(f"Saved {len(ds_sft)} SFT examples -> {JSONL_SFT_TRAIN}")

In [ ]:
import os
import random

NUM_GPUS     = int(os.environ.get("ASSN2_SFT_NUM_GPUS", os.environ.get("ASSN2_NUM_GPUS", "1")))
MASTER_PORT  = int(os.environ.get("ASSN2_MASTER_PORT", str(random.randint(20000, 40000))))
MAX_LEN      = int(os.environ.get("ASSN2_SFT_MAX_LEN", "4096"))
TRAIN_BATCH  = int(os.environ.get("ASSN2_SFT_TRAIN_BATCH_SIZE", "4"))
MICRO_BATCH  = int(os.environ.get("ASSN2_SFT_MICRO_BATCH_SIZE", "1"))
MAX_EPOCHS   = int(os.environ.get("ASSN2_SFT_MAX_EPOCHS", "2"))
ZERO_STAGE   = int(os.environ.get("ASSN2_SFT_ZERO_STAGE", "2"))
LOAD_4BIT    = os.environ.get("ASSN2_SFT_LOAD_IN_4BIT", "1") == "1"
LORA_RANK    = int(os.environ.get("ASSN2_SFT_LORA_RANK", "16" if LOAD_4BIT else "0"))
LORA_ALPHA   = int(os.environ.get("ASSN2_SFT_LORA_ALPHA", "32"))
LORA_DROPOUT = os.environ.get("ASSN2_SFT_LORA_DROPOUT", "0.0")
ATTN         = os.environ.get("ASSN2_ATTN_IMPLEMENTATION", "sdpa")
PARAM_DTYPE  = "bf16" if LOAD_4BIT else "fp16"

# ### EDIT HERE ### SFT learning rate.
SFT_LR = os.environ.get("ASSN2_SFT_ADAM_LR", "5e-6")

print(f"NUM_GPUS={NUM_GPUS}  LR={SFT_LR}  LORA_RANK={LORA_RANK}  EPOCHS={MAX_EPOCHS}  4bit={LOAD_4BIT}")

In [ ]:
import shlex

cmd = [
    "deepspeed", f"--num_gpus={NUM_GPUS}", "--master_port", str(MASTER_PORT),
    "--module", "openrlhf.cli.train_sft",
    "--model.model_name_or_path", MODEL_BASE_ID,
    "--data.dataset", str(JSONL_SFT_TRAIN),
    "--data.input_key", "problem", "--data.output_key", "solution",
    "--data.max_len", str(MAX_LEN),
    "--train.batch_size", str(TRAIN_BATCH), "--train.micro_batch_size", str(MICRO_BATCH),
    "--train.max_epochs", str(MAX_EPOCHS), "--train.seed", "42",
    "--logger.logging_steps", "1", "--eval.steps", "-1",
    "--ds.zero_stage", str(ZERO_STAGE), "--ds.param_dtype", PARAM_DTYPE,
    "--ds.attn_implementation", ATTN,
    "--adam.lr", SFT_LR,
    "--ckpt.output_dir", str(CKPT_SFT_LORA), "--ckpt.save_hf",
]
if LOAD_4BIT:
    cmd.append("--ds.load_in_4bit")
if LORA_RANK > 0:
    cmd += ["--ds.lora.rank", str(LORA_RANK), "--ds.lora.alpha", str(LORA_ALPHA),
            "--ds.lora.dropout", LORA_DROPOUT]

print("Running:", " ".join(shlex.quote(x) for x in cmd))
get_ipython().system(" ".join(shlex.quote(x) for x in cmd))

## 7. SFT — Merge LoRA, Evaluate, Upload

In [ ]:
import gc
import json
import torch
from pathlib import Path
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

candidates = sorted(
    [CKPT_SFT_LORA] + [p for p in CKPT_SFT_LORA.rglob("*") if p.is_dir()],
    key=lambda p: p.stat().st_mtime, reverse=True
)
adapter_dirs = [p for p in candidates if (p / "adapter_config.json").exists()]
hf_src = adapter_dirs[0] if adapter_dirs else CKPT_SFT_LORA

if not (Path(hf_src) / "adapter_config.json").exists():
    SFT_MERGED_DIR = Path(hf_src)
    print("No LoRA adapter — using checkpoint as full model:", SFT_MERGED_DIR)
else:
    cfg    = json.loads((Path(hf_src) / "adapter_config.json").read_text(encoding="utf-8"))
    base_id = cfg.get("base_model_name_or_path") or MODEL_BASE_ID
    tok_src = Path(hf_src) if (Path(hf_src) / "tokenizer_config.json").exists() else base_id
    tokenizer = AutoTokenizer.from_pretrained(tok_src, trust_remote_code=True)

    dtype    = torch.bfloat16 if torch.cuda.is_available() else torch.float32
    load_kw  = {"trust_remote_code": True, "torch_dtype": dtype,
                 "device_map": "auto" if torch.cuda.is_available() else {"": "cpu"}}
    base     = AutoModelForCausalLM.from_pretrained(base_id, **load_kw)
    merged   = PeftModel.from_pretrained(base, str(hf_src)).merge_and_unload()

    SFT_MERGED_DIR = CKPT_SFT_MERGED
    SFT_MERGED_DIR.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(SFT_MERGED_DIR, safe_serialization=True)
    tokenizer.save_pretrained(SFT_MERGED_DIR)

    del merged, base
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("Merged SFT model ->", SFT_MERGED_DIR)

In [ ]:
import gc, os, torch
from pathlib import Path

sft_eval_dir = Path(os.environ.get("ASSN2_SFT_EVAL_MODEL", str(globals().get("SFT_MERGED_DIR", CKPT_SFT_MERGED))))
if not sft_eval_dir.exists():
    raise FileNotFoundError(f"Merged SFT model not found: {sft_eval_dir}")

print("SFT eval checkpoint:", sft_eval_dir)
sft_acc = eval_model(str(sft_eval_dir), eval_examples)
print("sft_acc:", sft_acc)
if "base_acc" in globals():
    print("delta vs base:", sft_acc - base_acc)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, create_repo

sft_upload_dir = Path(globals().get("SFT_MERGED_DIR", CKPT_SFT_MERGED))
if not sft_upload_dir.is_dir() or not any(sft_upload_dir.glob("*.safetensors")):
    raise RuntimeError(f"Full merged model not found at {sft_upload_dir}. Run merge cell first.")

repo_id = os.getenv("HF_SFT_REPO_ID") or input("Hub repo id for SFT model (e.g. username/assn2-sft): ").strip()
if not repo_id or "/" not in repo_id:
    raise ValueError("repo_id must look like 'username/repo_name'")

create_repo(repo_id, repo_type="model", exist_ok=True)
HfApi().upload_folder(folder_path=str(sft_upload_dir), repo_id=repo_id, repo_type="model",
                       commit_message="Upload ASSN2 SFT merged model")
print("Uploaded ->", f"https://huggingface.co/{repo_id}")

## 8. DPO Training

In [ ]:
import os, random

os.environ.setdefault("PYTORCH_ALLOC_CONF", "expandable_segments:True")

PATH_DPO_INIT = os.environ.get("ASSN2_DPO_INIT_MODEL", str(CKPT_SFT_MERGED))
CKPT_DPO.mkdir(parents=True, exist_ok=True)

NUM_GPUS     = int(os.environ.get("ASSN2_NUM_GPUS", "4"))
MAX_LEN      = int(os.environ.get("ASSN2_MAX_LEN", "4096"))
ZERO_STAGE   = int(os.environ.get("ASSN2_ZERO_STAGE", "2"))
PARAM_DTYPE  = os.environ.get("ASSN2_PARAM_DTYPE", "bf16")
REF_OFFLOAD  = os.environ.get("ASSN2_REF_OFFLOAD", "1") == "1"
ADAM_OFFLOAD = os.environ.get("ASSN2_ADAM_OFFLOAD", "1") == "1"
ATTN_IMPL    = os.environ.get("ASSN2_ATTN_IMPLEMENTATION", "sdpa")
GRAD_CKPT    = os.environ.get("ASSN2_GRAD_CKPT", "1") == "1"

# ### EDIT HERE ### DPO hyperparameters.
DPO_LR          = os.environ.get("ASSN2_DPO_LR", "5e-7")
DPO_BETA        = os.environ.get("ASSN2_DPO_BETA", "0.25")
TRAIN_BATCH     = int(os.environ.get("ASSN2_TRAIN_BATCH_SIZE", "16"))
MICRO_BATCH     = int(os.environ.get("ASSN2_MICRO_BATCH_SIZE", "1"))

MASTER_PORT = int(os.environ.get("ASSN2_MASTER_PORT", str(random.randint(20000, 40000))))

min_global = NUM_GPUS * MICRO_BATCH
if TRAIN_BATCH < min_global:
    print(f"Raising TRAIN_BATCH to {min_global} (must be >= NUM_GPUS * MICRO_BATCH)")
    TRAIN_BATCH = min_global

print(f"LR={DPO_LR}  beta={DPO_BETA}  train_batch={TRAIN_BATCH}  micro={MICRO_BATCH}  gpus={NUM_GPUS}")
print(f"ref_offload={REF_OFFLOAD}  adam_offload={ADAM_OFFLOAD}  grad_ckpt={GRAD_CKPT}")

In [ ]:
import shlex, subprocess

try:
    out  = subprocess.check_output(["nvidia-smi","--query-gpu=index,memory.used","--format=csv,noheader,nounits"], text=True)
    rows = sorted([(int(a), int(b)) for a, b in (l.split(",") for l in out.strip().splitlines())], key=lambda t: t[1])
    DS_INCLUDE = "localhost:" + ",".join(str(idx) for idx, _ in rows[:NUM_GPUS])
    print("Auto-picked GPUs:", DS_INCLUDE)
except Exception as e:
    DS_INCLUDE = ""
    print("GPU auto-pick failed, using --num_gpus:", e)

cmd = ["deepspeed"]
cmd += (["--include", DS_INCLUDE] if DS_INCLUDE else [f"--num_gpus={NUM_GPUS}"])
cmd += ["--master_port", str(MASTER_PORT), "--module", "openrlhf.cli.train_dpo",
        "--model.model_name_or_path", PATH_DPO_INIT,
        "--ref.model_name_or_path",   PATH_DPO_INIT,
        "--data.dataset",    str(JSONL_PREF_DPO.resolve()),
        "--data.prompt_key", "prompt", "--data.chosen_key", "chosen", "--data.rejected_key", "rejected",
        "--ckpt.output_dir", str(CKPT_DPO), "--ckpt.save_hf",
        "--model.beta", DPO_BETA,
        "--train.max_epochs", "1",
        "--train.batch_size", str(TRAIN_BATCH), "--train.micro_batch_size", str(MICRO_BATCH),
        "--data.max_len", str(MAX_LEN),
        "--logger.logging_steps", "1",
        "--ds.zero_stage", str(ZERO_STAGE), "--ds.param_dtype", PARAM_DTYPE,
        "--adam.lr", DPO_LR,
        "--ds.attn_implementation", ATTN_IMPL]

if GRAD_CKPT:  cmd.append("--model.gradient_checkpointing_enable")
if REF_OFFLOAD:  cmd.append("--ref.offload")
if ADAM_OFFLOAD: cmd.append("--ds.adam_offload")

print("Command built. Run next cell to launch.")

In [ ]:
import shlex

cmd_str = " ".join(shlex.quote(x) for x in cmd)
print("Running:", cmd_str)
get_ipython().system(cmd_str)

## 9. DPO — Evaluate and Upload

In [ ]:
import gc, os, torch
from pathlib import Path

ckpt_tuned = Path(os.environ.get("ASSN2_TUNED_OUT", str(CKPT_DPO)))
candidates = sorted([ckpt_tuned] + [p for p in ckpt_tuned.rglob("*") if p.is_dir()],
                     key=lambda p: p.stat().st_mtime, reverse=True)
adapter_dirs = [p for p in candidates if (p/"adapter_config.json").exists() and (p/"config.json").exists()]
full_dirs    = [p for p in candidates if (p/"config.json").exists() and not (p/"adapter_config.json").exists()]
hf_model_dir = adapter_dirs[0] if adapter_dirs else (full_dirs[0] if full_dirs else None)

if hf_model_dir is None:
    raise FileNotFoundError(f"No HF checkpoint found under {ckpt_tuned}. Run DPO with --ckpt.save_hf.")

print("DPO model:", hf_model_dir)
tuned_acc = eval_model(str(hf_model_dir), eval_examples)
print("tuned_acc:", tuned_acc)
if "base_acc" in globals():
    print("delta vs base:", tuned_acc - base_acc)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

In [ ]:
import gc, json, os
import torch
from pathlib import Path
from huggingface_hub import HfApi, create_repo
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

ckpt_tuned = Path(os.environ.get("ASSN2_TUNED_OUT", str(CKPT_DPO)))
candidates = sorted([ckpt_tuned] + [p for p in ckpt_tuned.rglob("*") if p.is_dir()],
                     key=lambda p: p.stat().st_mtime, reverse=True)
adapter_dirs = [p for p in candidates if (p/"adapter_config.json").exists() and (p/"config.json").exists()]
full_dirs    = [p for p in candidates if (p/"config.json").exists() and not (p/"adapter_config.json").exists()]
hf_src = adapter_dirs[0] if adapter_dirs else (full_dirs[0] if full_dirs else None)
if hf_src is None:
    raise FileNotFoundError(f"No DPO checkpoint found under {ckpt_tuned}")

if (hf_src / "adapter_config.json").exists():
    cfg     = json.loads((hf_src / "adapter_config.json").read_text(encoding="utf-8"))
    base_id = cfg.get("base_model_name_or_path") or MODEL_BASE_ID
    tok     = AutoTokenizer.from_pretrained(hf_src if (hf_src/"tokenizer_config.json").exists() else base_id, trust_remote_code=True)
    load_kw = {"trust_remote_code": True, "torch_dtype": torch.bfloat16 if torch.cuda.is_available() else torch.float32,
                "device_map": "auto" if torch.cuda.is_available() else {"": "cpu"}}
    base    = AutoModelForCausalLM.from_pretrained(base_id, **load_kw)
    merged  = PeftModel.from_pretrained(base, str(hf_src)).merge_and_unload()
    upload_dir = Path(os.environ.get("ASSN2_DPO_MERGED_OUT", str(CKPT_DPO_MERGED)))
    upload_dir.mkdir(parents=True, exist_ok=True)
    merged.save_pretrained(upload_dir, safe_serialization=True)
    tok.save_pretrained(upload_dir)
    del merged, base; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print("Merged ->", upload_dir)
else:
    upload_dir = hf_src
    print("Full model, no merge needed:", upload_dir)

repo_id = os.getenv("HF_REPO_ID") or input("Hub repo id for DPO model (e.g. username/assn2-dpo): ").strip()
if not repo_id or "/" not in repo_id:
    raise ValueError("repo_id must look like 'username/repo_name'")

create_repo(repo_id, repo_type="model", exist_ok=True)
HfApi().upload_folder(folder_path=str(upload_dir), repo_id=repo_id, repo_type="model",
                       commit_message="Upload ASSN2 DPO merged model")
print("Uploaded ->", f"https://huggingface.co/{repo_id}")

## 10. Upload Preference Dataset to Hub

In [ ]:
import os
from pathlib import Path
from huggingface_hub import HfApi, create_repo

pref_file = Path(globals().get("JSONL_PREF_DPO", DATA_DIR / "assn2_pref_dpo.jsonl"))
if not pref_file.is_file():
    raise FileNotFoundError(f"Missing {pref_file}. Run the preference-generation section first.")

pref_repo = os.getenv("HF_PREF_REPO_ID") or input("Hub dataset repo id (e.g. username/assn2-prefs): ").strip()
if not pref_repo or "/" not in pref_repo:
    raise ValueError("Repo id must look like 'username/repo_name'")

create_repo(pref_repo, repo_type="dataset", exist_ok=True)
HfApi().upload_file(path_or_fileobj=str(pref_file), path_in_repo="assn2_pref.jsonl",
                     repo_id=pref_repo, repo_type="dataset",
                     commit_message="Add ASSN2 synthetic DPO preference JSONL")
print("Uploaded ->", f"https://huggingface.co/datasets/{pref_repo}")

## 11. SimPO Pipeline

> **EDIT HERE** — Implement the full SimPO pipeline in this section.
>
> Suggested steps:
> 1. Add/modify the SimPO trainer logic in OpenRLHF (`third_party/OpenRLHF/openrlhf/trainer/`)
> 2. Build a DeepSpeed launch command similar to the DPO cell (point to `openrlhf.cli.train_simpo` or your custom entry point)
> 3. Merge the SimPO LoRA adapter
> 4. Evaluate with `eval_model(str(CKPT_SIMPO_MERGED), eval_examples)`
> 5. Upload to the Hub and submit to the leaderboard

In [ ]:
# ### EDIT HERE ### SimPO pipeline
# Implement SimPO training + evaluation in this section.